# `braincell.quad` — Numerical Integrators for Differential Equations

`braincell.quad` is the numerical-integration toolkit that powers every
neuron model in BrainCell. It provides a small, decoupled set of
abstractions for describing differential equations and a curated catalog
of single-step integrators (Runge–Kutta, exponential Euler, backward
Euler, implicit splitting, voltage solvers, …) that can be plugged in
interchangeably wherever an integrator is required.

This notebook walks through:

1. [What `braincell.quad` is for](#1.-What-braincell.quad-is-for)
2. [Key abstractions](#2.-Key-abstractions)
3. [Setup and installation](#3.-Setup-and-installation)
4. [A first end-to-end example](#4.-A-first-end-to-end-example)
5. [The integrator catalog](#5.-The-integrator-catalog)
6. [Looking integrators up at runtime](#6.-Looking-integrators-up-at-runtime)
7. [Registering your own integrator](#7.-Registering-your-own-integrator)
8. [Independent sub-system integration](#8.-Independent-sub-system-integration)
9. [Best practices](#9.-Best-practices)
10. [Further reading](#10.-Further-reading)


## 1. What `braincell.quad` is for

Every BrainCell neuron exposes a system of ordinary differential
equations (ODEs):

$$\frac{dy}{dt} = f(t, y)$$

where $y$ is the concatenation of every membrane voltage, every gating
variable, every ion concentration, and every other dynamic state in the
cell. `braincell.quad` is responsible for advancing those ODEs **one
time step at a time** in a way that is:

- **Modular** — the integrator is decoupled from the model. Switching
  between forward Euler, RK4, exponential Euler, backward Euler, an
  operator-splitting scheme, or a diffrax-backed implicit method is a
  one-line change.
- **JAX-native** — every integrator is differentiable, JIT-compatible,
  and runs on CPU/GPU/TPU through `jax.numpy`. There are no Python
  callbacks in the inner loop.
- **Unit-aware** — all state values, derivatives, and time steps carry
  `brainunit` quantities and the integrators check dimensional
  consistency for you.
- **Discoverable** — every step function is registered in a process-wide
  registry, so users (and tooling) can introspect what is available, ask
  for an integrator by name, or wire up new methods without touching the
  package's `__init__.py`.

If you just want to *use* a neuron, you usually never call anything in
`braincell.quad` directly — you pass the integrator name as the
`solver=` argument to a model class such as `SingleCompartment` or
`Cell`. The rest of this notebook is for users who want to understand
*how* that works, swap integrators, write a new one, or test
integrator-level code in isolation.


## 2. Key abstractions

`braincell.quad` is intentionally tiny. There are exactly four moving
parts you need to understand.

### 2.1 `DiffEqState`

A `DiffEqState` is a `brainstate.HiddenState` subclass that announces
"this state is integrated by an ODE/SDE solver". On top of the usual
`value` field it carries two extra slots:

- `derivative` — the right-hand side `dy/dt`. Set by your model in
  `compute_derivative`. Used by every integrator.
- `diffusion` — optional Wiener-process diffusion term `g(t, y)` for
  stochastic models. Most integrators ignore it; SDE-aware solvers
  consume it.

A `DiffEqState` is the *contract* between your model and `quad`: any
state you want integrated must be wrapped in one. Plain
`brainstate.State` / `ShortTermState` instances are passed through
unchanged by the integrator step.

In [1]:
import brainunit as u
import jax.numpy as jnp
from braincell.quad import DiffEqState

V = DiffEqState(jnp.full((3,), -65.0) * u.mV)        # value
V.derivative = jnp.zeros(3) * u.mV / u.ms            # set in compute_derivative

print("V.value      =", V.value)
print("V.derivative =", V.derivative)
print("V.diffusion  =", V.diffusion)  # None until you set it


V.value      = [-65. -65. -65.] mV
V.derivative = [0. 0. 0.] mV / ms
V.diffusion  = None


### 2.2 `DiffEqModule`

A mixin that any model class wanting to be integrated must inherit
from. It defines three hook methods:

| Method                   | When it's called                                          | Default      |
|--------------------------|-----------------------------------------------------------|--------------|
| `pre_integral(*a)`       | Once at the start of every step                           | no-op        |
| `compute_derivative(*a)` | One or more times per step (depending on the integrator) | **must override** |
| `post_integral(*a)`      | Once at the end of every step                             | no-op        |

`compute_derivative` is the only method you must override — it should
read the current `value` of every `DiffEqState` and set its
`derivative`. The integrator will call it as many times as the method
needs (e.g. four times for RK4) inside `brainstate.StateTraceStack`
contexts so that it can capture which states were touched.

In [2]:
import brainstate
import brainunit as u
import jax.numpy as jnp
from braincell.quad import DiffEqState, DiffEqModule


class LinearDecay(brainstate.nn.Module, DiffEqModule):
    """dx/dt = -x / tau."""

    def __init__(self, x0=1.0, tau=10.0 * u.ms):
        super().__init__()
        self.tau = tau
        self.x = DiffEqState(jnp.full((3,), x0) * u.mV)

    def compute_derivative(self):
        self.x.derivative = -self.x.value / self.tau


m = LinearDecay()
print("Initial value:", m.x.value)
m.compute_derivative()
print("Derivative   :", m.x.derivative)


Initial value: [1. 1. 1.] mV
Derivative   : [-0.1 -0.1 -0.1] mV / ms


This four-line class is enough to be driven by *any* of the explicit RK
solvers, the backward Euler solver, the independent exponential Euler
solver, and the implicit Euler solver.

### 2.3 `IndependentIntegration`

An additional mixin that flags a sub-module as having its **own**
integrator. When the parent's integrator builds the integrated-state
list, every state living under an `IndependentIntegration` sub-module is
excluded. The sub-module is then expected to integrate its own state
(typically by calling `self.solver(self, *args)` from `update`).

This is the mechanism BrainCell uses for things like calcium dynamics
that should advance with a different scheme — or a different time
step — from the main membrane integrator. See section 8 for a worked
example.

### 2.4 The integrator step function

Every integrator in `braincell.quad` is a function with the canonical
shape:

```python
def some_step(target: DiffEqModule, *args):
    ...
```

It mutates `target` in place: every `DiffEqState.value` is replaced with
its value at `t + dt`. The current time `t` and step size `dt` are read
from the `brainstate.environ` context, **not** passed as arguments.
A few legacy integrators (`implicit_euler_step`, `splitting_step`,
`cn_*_step`, `implicit_rk4_step`, `dhs_voltage_step`) take `t` and `dt`
as explicit positional arguments — those are clearly marked in their
signatures.

The `*args` are forwarded to `pre_integral`, `compute_derivative`, and
`post_integral` so the model can receive things like an external current
input on every step.

## 3. Setup and installation

`braincell.quad` is part of the main `braincell` package — there is
nothing extra to install. The development install gives you everything
you need:

```bash
git clone https://github.com/...braincell.git
cd braincell
pip install -e ".[testing]"
```

Optional dependencies:

- **diffrax** — Enables the `diffrax_*_step` family (Tsit5, Dopri5,
  Dopri8, Kvaerno3/4/5, …). Install with `pip install diffrax`. The
  diffrax-backed step functions are imported lazily, so you do not pay
  for diffrax's transitive dependency tree unless you actually call one
  of them.

`braincell.quad` runs on CPU by default but transparently uses
GPU/TPU through JAX if a device is visible. The cell below verifies
that your install is working.

In [3]:
import braincell.quad as quad
print("Available canonical integrators:")
for name in quad.get_registry().names():
    print(f"  - {name}")
print(f"\nTotal: {len(quad.get_registry())} canonical, "
      f"{len(quad.all_integrators)} including aliases.")


Available canonical integrators:
  - backward_euler
  - cn_exp_euler
  - cn_rk4
  - dhs_voltage
  - euler
  - exp_euler
  - exp_exp_euler
  - heun2
  - heun3
  - implicit_euler
  - implicit_exp_euler
  - implicit_rk4
  - ind_exp_euler
  - midpoint
  - ralston2
  - ralston3
  - ralston4
  - rk2
  - rk3
  - rk4
  - splitting
  - ssprk3
  - staggered

Total: 23 canonical, 25 including aliases.


## 4. A first end-to-end example

Here is a complete script that integrates a linear decay ODE with
several different solvers and compares their accuracy against the
analytical solution $x(t) = e^{-t/\tau}$.

In [4]:
import math
import brainstate
import brainunit as u
import jax.numpy as jnp

from braincell.quad import DiffEqState, DiffEqModule, get_integrator


class LinearDecay(brainstate.nn.Module, DiffEqModule):
    def __init__(self, x0=1.0, tau=10.0 * u.ms):
        super().__init__()
        self.tau = tau
        self.x = DiffEqState(jnp.asarray(x0) * u.mV)

    def compute_derivative(self):
        self.x.derivative = -self.x.value / self.tau


def run(method_name: str, dt=0.1 * u.ms, n_steps=100):
    model = LinearDecay()
    step = get_integrator(method_name)         # resolve string -> function
    with brainstate.environ.context(dt=dt):
        for i in range(n_steps):
            with brainstate.environ.context(t=i * dt):
                step(model)
    return float(model.x.value.to_decimal(u.mV))


target = math.exp(-1.0)  # analytical x(10 ms) for tau = 10 ms
print(f"{'method':>16s}  {'final':>12s}  {'error':>12s}")
print("-" * 44)
for name in ["euler", "rk4", "backward_euler", "ind_exp_euler"]:
    final = run(name)
    err = abs(final - target)
    print(f"{name:>16s}  {final:>12.7f}  {err:>12.2e}")


          method         final         error
--------------------------------------------
           euler     0.3660323      1.85e-03
             rk4     0.3678796      1.58e-07
  backward_euler     0.3697112      1.83e-03
   ind_exp_euler     0.3678796      1.58e-07


The qualitative pattern is exactly what theory predicts:

- `euler` is **first-order** — error scales linearly with `dt`.
- `rk4` is **fourth-order** — many orders of magnitude better.
- `backward_euler` is also first-order but **L-stable** — it never goes
  unstable, even for stiff systems.
- `ind_exp_euler` exactly integrates a locally linear ODE
  ($dx/dt = \lambda x$) in a single step, hence the near-machine-epsilon
  error.

Note three things about the script:

1. **The model code never mentions a particular solver.** The integrator
   is resolved at runtime via `get_integrator(name)`.
2. **`t` and `dt` are passed via `brainstate.environ.context(...)`.**
   This is how every integrator inside `quad` discovers them.
3. **All quantities carry brainunit units.** The integrator validates
   the dimensions of `state.value`, `state.derivative`, and `dt` and
   raises if they are inconsistent.

## 5. The integrator catalog

`braincell.quad` ships with five families of step functions. The full
list, keyed by category, is also available programmatically via
`get_registry().by_category(...)`.

### 5.1 Explicit Runge–Kutta (`category="explicit"`)

| Canonical name | Order | Description                                      |
|----------------|-------|--------------------------------------------------|
| `euler`        |   1   | Forward Euler. Aliased as `explicit`.            |
| `midpoint`     |   2   | Modified-Euler / midpoint method.                |
| `rk2`          |   2   | Generic 2-stage method.                          |
| `heun2`        |   2   | Heun's second-order predictor-corrector.         |
| `ralston2`     |   2   | Ralston's minimum-truncation 2-stage.            |
| `rk3`          |   3   | Classical third-order Runge–Kutta.               |
| `heun3`        |   3   | Heun's third-order method.                       |
| `ssprk3`       |   3   | Strong-stability-preserving RK3 (Shu–Osher).     |
| `ralston3`     |   3   | Ralston's 3-stage minimum-truncation method.     |
| `rk4`          |   4   | Classical 4-stage RK4.                           |
| `ralston4`     |   4   | Ralston's 4-stage method.                        |

Use these as your default for non-stiff problems. `rk4` is a good
all-rounder; `ssprk3` is the right choice for problems with sharp
gradients.

### 5.2 Exponential integrators (`category="exponential"`)

| Canonical name      | Order | Notes                                    |
|---------------------|-------|------------------------------------------|
| `exp_euler`         |   1   | Coupled exponential Euler. Linearizes the entire state vector around the current point. Requires an `HHTypedNeuron` target. |
| `ind_exp_euler`     |   1   | Independent exponential Euler — linearizes each `DiffEqState` separately. The default for HH gating variables. |
| `exp_exp_euler`     |   1   | Cell-only: exponential axial-current solve combined with exponential gating updates. |

Exponential integrators are the gold standard for HH-style ion-channel
gating equations because they are exact for locally linear dynamics
and stable at much larger step sizes than explicit methods.

### 5.3 Implicit / Newton-based (`category="implicit"`)

| Canonical name        | Order | Notes                                                    |
|-----------------------|-------|----------------------------------------------------------|
| `backward_euler`      |   1   | Implicit Euler via local Jacobian linearization.         |
| `implicit_euler`      |   1   | Newton iteration on the implicit Euler residual.         |
| `splitting`           |   –   | Operator-split: implicit axial currents + Newton gating. |
| `implicit_rk4`        |   4   | Implicit axial currents + explicit RK4 gating.           |
| `implicit_exp_euler`  |   –   | Implicit axial currents + exponential Euler gating.      |
| `cn_rk4`              |   –   | Crank–Nicolson axial + explicit RK4 gating.              |
| `cn_exp_euler`        |   –   | Crank–Nicolson axial + exponential Euler gating.         |

These are the right family for stiff multi-compartment cable equations.
Most of them only run on `Cell` targets (not `SingleCompartment`)
because they assemble an axial-current matrix from the morphology.

### 5.4 Voltage solver (`category="voltage"`)

| Canonical name | Notes                                                                                                                                |
|----------------|--------------------------------------------------------------------------------------------------------------------------------------|
| `dhs_voltage`  | Implicit-Euler **D**ynamic **H**ierarchical **S**olver on the active point-tree runtime. Runs the linear voltage solve only — gating is handled separately. |

`dhs_voltage_step` is the building block of the `staggered` integrator
below.

### 5.5 Staggered (`category="staggered"`)

| Canonical name | Aliases    | Notes                                                          |
|----------------|------------|----------------------------------------------------------------|
| `staggered`    | `stagger`  | DHS implicit voltage solve followed by `ind_exp_euler` for ion channels. The recommended default for active multi-compartment cells. |

### 5.6 diffrax-backed (`category="diffrax"`, optional)

When `diffrax` is installed, the registry also exposes:

| Canonical name      | Order | Backing solver                |
|---------------------|-------|-------------------------------|
| `diffrax_euler`     |  1    | `diffrax.Euler`               |
| `diffrax_heun`      |  2    | `diffrax.Heun`                |
| `diffrax_midpoint`  |  2    | `diffrax.Midpoint`            |
| `diffrax_ralston`   |  2    | `diffrax.Ralston`             |
| `diffrax_bosh3`     |  3    | `diffrax.Bosh3`               |
| `diffrax_tsit5`     |  5    | `diffrax.Tsit5`               |
| `diffrax_dopri5`    |  5    | `diffrax.Dopri5`              |
| `diffrax_dopri8`    |  8    | `diffrax.Dopri8`              |
| `diffrax_bwd_euler` |  1    | `diffrax.ImplicitEuler`       |
| `diffrax_kvaerno3`  |  3    | `diffrax.Kvaerno3` (implicit) |
| `diffrax_kvaerno4`  |  4    | `diffrax.Kvaerno4` (implicit) |
| `diffrax_kvaerno5`  |  5    | `diffrax.Kvaerno5` (implicit) |

If `diffrax` is *not* installed, the `diffrax_*_step` functions are
still importable from `braincell.quad` but they are not registered, and
calling `get_integrator("diffrax_dopri5")` raises `ValueError` with a
helpful suggestion.

In [5]:
# Browse the registry by category at runtime.
from braincell.quad import get_registry

registry = get_registry()
for category in ["explicit", "exponential", "implicit", "voltage", "staggered"]:
    entries = registry.by_category(category)
    if not entries:
        continue
    print(f"## {category}")
    for entry in entries:
        order = "" if entry.order is None else f" (order {entry.order})"
        print(f"  - {entry.name}{order}: {entry.description}")
    print()


## explicit
  - euler (order 1): Forward (explicit) Euler method.
  - heun2 (order 2): Heun's second-order Runge-Kutta method.
  - heun3 (order 3): Heun's third-order Runge-Kutta method.
  - midpoint (order 2): Explicit midpoint (modified Euler) method.
  - ralston2 (order 2): Ralston's second-order Runge-Kutta method.
  - ralston3 (order 3): Ralston's third-order Runge-Kutta method.
  - ralston4 (order 4): Ralston's fourth-order Runge-Kutta method.
  - rk2 (order 2): Generic second-order Runge-Kutta method.
  - rk3 (order 3): Classical third-order Runge-Kutta method.
  - rk4 (order 4): Classical four-stage fourth-order Runge-Kutta method.
  - ssprk3 (order 3): Strong-stability-preserving third-order Runge-Kutta (SSPRK3).

## exponential
  - exp_euler (order 1): Coupled exponential Euler step linearizing the full state vector.
  - exp_exp_euler: Exponential axial integration paired with exponential Euler gating updates.
  - ind_exp_euler (order 1): Independent exponential Euler step (p

## 6. Looking integrators up at runtime

There are three equivalent ways to find a step function:

```python
from braincell.quad import get_integrator, get_registry, all_integrators

# 1. By canonical name or alias.
step = get_integrator("rk4")           # canonical
step = get_integrator("explicit")      # alias for "euler"
step = get_integrator("stagger")       # alias for "staggered"

# 2. By passing through a callable unchanged.
step = get_integrator(my_custom_step)  # returns my_custom_step
```

`get_integrator` raises `ValueError` for unknown names and includes a
"Did you mean …?" suggestion based on `difflib`.

In [6]:
from braincell.quad import get_integrator, all_integrators

# Canonical and alias resolution.
print(get_integrator("rk4"))
print(get_integrator("explicit") is get_integrator("euler"))
print(get_integrator("stagger") is get_integrator("staggered"))

# Did-you-mean suggestion on a typo.
try:
    get_integrator("eler")
except ValueError as e:
    print("ValueError:", e)


<function rk4_step at 0x0000023275A90C20>
True
True
ValueError: Unknown integrator 'eler'. Did you mean 'euler'? Available: backward_euler, cn_exp_euler, cn_rk4, dhs_voltage, euler, exp_euler, exp_exp_euler, heun2, heun3, implicit_euler, implicit_exp_euler, implicit_rk4, ind_exp_euler, midpoint, ralston2, ralston3, ralston4, rk2, rk3, rk4, splitting, ssprk3, staggered.


The `all_integrators` mapping is a backwards-compatible read-only view
that includes alias keys.

In [7]:
print("'explicit' in all_integrators:", "explicit" in all_integrators)
print("Same function as 'euler':", all_integrators["explicit"] is all_integrators["euler"])

# It is read-only — assignment fails.
try:
    all_integrators["euler"] = lambda *a, **k: None
except TypeError as e:
    print("Read-only:", e)


'explicit' in all_integrators: True
Same function as 'euler': True
Read-only: '_RegistryDictView' object does not support item assignment


For richer introspection, use the registry directly:

In [8]:
from braincell.quad import get_registry

registry = get_registry()
print("canonical names:", registry.names()[:6], "...")
print("rk4 entry      :", registry.entry("rk4"))
print("rk4 description:", registry.entry("rk4").description)


canonical names: ['backward_euler', 'cn_exp_euler', 'cn_rk4', 'dhs_voltage', 'euler', 'exp_euler'] ...
rk4 entry      : IntegratorEntry(name='rk4', func=<function rk4_step at 0x0000023275A90C20>, aliases=(), category='explicit', order=4, description='Classical four-stage fourth-order Runge-Kutta method.', deprecated=False, module='braincell.quad._runge_kutta')
rk4 description: Classical four-stage fourth-order Runge-Kutta method.


## 7. Registering your own integrator

Adding a new integrator is a one-decorator job. Drop the file in
`braincell/quad/`, decorate it with `@register_integrator`, and import
it from `braincell/quad/__init__.py` so the decorator runs at package
import time.

For demonstration we register a toy "scaled Euler" integrator on the
fly into the live registry.

In [9]:
import brainstate
from braincell._misc import set_module_as
from braincell.quad import (
    DiffEqModule,
    register_integrator,
    get_integrator,
    get_registry,
)
from braincell.quad._util import apply_standard_solver_step


def _scaled_euler_solver(f, y0, t, dt, args=()):
    """Forward Euler with a 0.5 stability damping factor."""
    dy, _aux = f(t, y0, *args)
    return y0 + 0.5 * dt * dy, {}


@register_integrator(
    "demo_scaled_euler",
    aliases=("demo_euler",),
    category="explicit",
    order=1,
    description="Tutorial-only forward Euler with a 0.5 damping factor.",
    override=True,        # allow re-running this cell without crashing
)
@set_module_as("braincell")
def demo_scaled_euler_step(target, *args):
    t = brainstate.environ.get("t")
    dt = brainstate.environ.get("dt")
    apply_standard_solver_step(_scaled_euler_solver, target, t, dt, *args)


# Confirm registration.
print("registered as     :", get_registry().entry("demo_scaled_euler"))
print("canonical resolves:", get_integrator("demo_scaled_euler") is demo_scaled_euler_step)
print("alias resolves    :", get_integrator("demo_euler") is demo_scaled_euler_step)


registered as     : IntegratorEntry(name='demo_scaled_euler', func=<function demo_scaled_euler_step at 0x00000232095AC2C0>, aliases=('demo_euler',), category='explicit', order=1, description='Tutorial-only forward Euler with a 0.5 damping factor.', deprecated=False, module='__main__')
canonical resolves: True
alias resolves    : True


A few rules to keep in mind:

- **Place `@register_integrator` *outside* `@set_module_as`** so the
  registry stores the final exported function.
- **Names must be globally unique.** Re-registering an existing name
  raises `ValueError` unless you pass `override=True`, which also emits
  a `RuntimeWarning` identifying the previous owner.
- **Aliases must not collide** with any other canonical name or alias.
- **Set `category` and `order`** when relevant — they show up in
  documentation and `by_category` lookups.

If you ever need to register a method dynamically (not via the
decorator), use `get_registry().register(...)` directly.

Now clean up the demo registration so the rest of the notebook keeps
the registry tidy:

In [10]:
get_registry().unregister("demo_scaled_euler")
print("demo_scaled_euler in registry:", "demo_scaled_euler" in get_registry())


demo_scaled_euler in registry: False


## 8. Independent sub-system integration

Sometimes a sub-module wants to integrate itself with a different
scheme or a different step size from the rest of the model. For
example, a calcium-buffer pool might use a stiff exponential Euler
update while the rest of the cell uses RK4.

Make the sub-module inherit from both `DiffEqModule` and
`IndependentIntegration`. The constructor takes a `solver=` argument
(string or callable) that will be cached on the instance. Then drive it
manually inside the parent's `update`.

In [11]:
import brainstate
import brainunit as u
import jax.numpy as jnp
from braincell.quad import (
    DiffEqState,
    DiffEqModule,
    IndependentIntegration,
    rk4_step,
)


class CalciumPool(brainstate.nn.Module, DiffEqModule, IndependentIntegration):
    def __init__(self):
        IndependentIntegration.__init__(self, solver="ind_exp_euler")
        brainstate.nn.Module.__init__(self)
        self.Ca = DiffEqState(jnp.zeros(()) * u.mM)

    def compute_derivative(self, *args):
        self.Ca.derivative = -self.Ca.value / (50.0 * u.ms)


class Cell(brainstate.nn.Module, DiffEqModule):
    def __init__(self):
        super().__init__()
        self.V = DiffEqState(-65.0 * u.mV)
        self.calcium = CalciumPool()           # owns its own integrator

    def compute_derivative(self, I_ext):
        # The calcium state will NOT appear here because it lives under
        # an IndependentIntegration sub-module — split_diffeq_states
        # excludes it from the integration set.
        self.V.derivative = I_ext / (1.0 * u.uF)

    def update(self, I_ext):
        # Step the main system, then advance the sub-system separately.
        rk4_step(self, I_ext)
        self.calcium.make_integration()


# Verify split_diffeq_states excludes ``calcium.Ca`` from the parent.
from braincell.quad._util import split_diffeq_states

cell = Cell()
all_st, diffeq_st, other_st = split_diffeq_states(cell)
print("All states     :", sorted(all_st.keys()))
print("Integrated here:", sorted(diffeq_st.keys()))


All states     : [('V',), ('calcium', 'Ca')]
Integrated here: [('V',)]


Under the hood `make_integration` calls `self.solver(self, *args)`, so
the sub-module is a regular target from the integrator's point of view.

## 9. Best practices

A short list of habits that will save you time:

- **Resolve solvers by name in your model's `__init__`, not at every
  step.** `self.solver = get_integrator(method)` once is faster, and it
  surfaces typos at construction time instead of inside a JIT trace.
- **Use units consistently.** Every `DiffEqState`, derivative, and `dt`
  must carry brainunit quantities. The integrators check dimensional
  consistency via `_check_diffeq_state_derivative` and raise on
  mismatches.
- **Pick the integrator that matches your stiffness regime.**
  - Non-stiff (passive cables, simple LIF): `rk4` or `midpoint`.
  - HH single compartments: `ind_exp_euler` is the de-facto default.
  - Multi-compartment HH cells: `staggered` (DHS + ind_exp_euler).
  - Stiff cable + custom gating: `cn_exp_euler` or `implicit_exp_euler`.
  - When in doubt, run a `dt`-refinement convergence study like the one
    in `_runge_kutta_test.py::RungeKuttaConvergenceTest`.
- **Don't put solver state on `brainstate.State` outside `DiffEqState`.**
  Plain states are passed through unchanged by the integrator, so
  anything you store on them will be silently constant across the step.
- **Wrap stochastic dynamics in `DiffEqState.diffusion`**, not by
  injecting `jax.random.normal` calls inside `compute_derivative` —
  that breaks the multi-stage RK methods because each stage would draw
  fresh randomness.
- **Use `brainstate.environ.context(t=..., dt=...)` to drive simulation
  loops.** Every integrator reads `t`/`dt` from the environ; setting
  them once per step keeps the call site clean.
- **Use `IndependentIntegration` for time-scale-separated subsystems**
  rather than custom hacks inside `compute_derivative`.
- **When writing a new integrator**, prefer to delegate the
  state-management plumbing to `apply_standard_solver_step`. Your job
  is just to implement the inner `(f, y0, t, dt, args) → (y1, aux)`
  step function on dimensionless arrays.
- **Let the registry do the bookkeeping.** Never scan `locals()` or
  hard-code a `{"euler": euler_step, ...}` dict; ask the registry.